# Run your Object ReID experiment
Run cells in order on a Colab GPU. This uses the cleaned local source bundle, not the older GitHub branch.

Upload `obj_reid_colab.zip` and your existing `images.tar.gz` to **My Drive / ObjReID_Run** before starting. The ZIP contains source, original metadata and recovered checkpoints; the image archive stays separate.

The recovered checkpoint remains an original artifact. New training uses a **new seeded, identity-disjoint development experiment**, not a reconstruction of the lost course split. Compare models on this new subset and label it accordingly. Do not use the historical checkpoint's accuracy as verified held-out performance.

Choose Runtime > Change runtime type > GPU. Run each cell manually and inspect its output. Stop on errors. GPU availability and session limits vary. Source code is copied into each run's Drive backup so later edits do not change its provenance.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime
import json, shutil, zipfile, subprocess, sys

drive.mount('/content/drive')
INPUT = Path('/content/drive/MyDrive/ObjReID_Run')
assert (INPUT / 'obj_reid_colab.zip').is_file(), 'Upload obj_reid_colab.zip to My Drive/ObjReID_Run'
assert (INPUT / 'images.tar.gz').is_file(), 'Upload images.tar.gz to the same folder'
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
BACKUP = INPUT / 'runs' / RUN_ID
BACKUP.mkdir(parents=True, exist_ok=False)
PROJECT = Path('/content') / ('obj_reid_' + RUN_ID)
PROJECT.mkdir()
with zipfile.ZipFile(INPUT / 'obj_reid_colab.zip') as archive:
    for member in archive.infolist():
        assert (PROJECT / member.filename).resolve().is_relative_to(PROJECT.resolve())
    archive.extractall(PROJECT)
shutil.copy2(INPUT / 'obj_reid_colab.zip', BACKUP / 'source_and_original_artifacts.zip')
import os
os.chdir(PROJECT)
print('New run:', RUN_ID)
print('Persistent backup:', BACKUP)

## Install dependencies and check GPU
Keep Colab's existing GPU-enabled PyTorch. This notebook intentionally does not run `uv sync`, which could replace it with another build. The exact installed environment is saved for this run.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'open-clip-torch==3.3.0', 'timm==1.0.26', 'pyarrow', 'pandas', 'scikit-learn', 'scipy', 'pyyaml', 'tqdm', 'pillow', 'pytest'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--no-build-isolation', '-e', '.'], check=True)
(BACKUP / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
(BACKUP / 'hardware.json').write_text(json.dumps({'gpu':torch.cuda.get_device_name(0), 'torch':str(torch.__version__), 'cuda':torch.version.cuda, 'python':sys.version}, indent=2))

def run(*args):
    command = [sys.executable, '-u', *map(str, args)]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait() != 0:
            raise subprocess.CalledProcessError(process.returncode, command)

run('-m', 'pytest', '-q')

## Recover images on the runtime disk
Copy the large archive once from Drive, then extract locally for faster training. This can take several minutes and uses additional disk space. The original metadata remains unchanged. The archive contains about 5.7 GB of referenced uncompressed images.

In [ ]:
print('Copying image archive from Drive...')
shutil.copy2(INPUT / 'images.tar.gz', PROJECT / 'images.tar.gz')
run('scripts/prepare_dataset.py')
shutil.copy2('results/dataset_a_recovery.json', BACKUP / 'dataset_a_recovery.json')

## Make and save the new experiment split
Seed 42 selects 800 training, 100 validation and 100 test identities from identities with at least three images. Each identity contributes up to eight images. All images belonging to an identity stay in one partition. This is a manageable **subset experiment**, not the original course benchmark. Training and validation identities come only from the supplied training metadata. Test identities come from the supplied test metadata after excluding every identity present in training metadata. This avoids training on the candidate held-out identities.

Do not change the split or tune settings after inspecting the test results. Validation is reserved for development; the example uses fixed training settings without validation-based checkpoint selection.

In [ ]:
import pandas as pd
import numpy as np
from reid_data import validate_images

SEED = 42
TRAIN_IDS, VAL_IDS, TEST_IDS, MAX_IMAGES = 800, 100, 100, 8
original = PROJECT / 'datasets/dataset_a'
training_source = pd.read_parquet(original / 'train.parquet')
test_source = pd.read_parquet(original / 'test.parquet')
pool = pd.concat([training_source, test_source], ignore_index=True)
for column in ['identity', 'camera_id']:
    assert pool.groupby('image_path')[column].nunique().max() == 1, 'Conflicting source labels'
pool = pool.drop_duplicates('image_path').sort_values(['identity', 'image_path']).reset_index(drop=True)
counts = training_source.groupby('identity').size()
eligible_training = sorted(counts[counts >= 3].index.tolist())
heldout_source = test_source[~test_source.identity.isin(training_source.identity)]
heldout_counts = heldout_source.groupby('identity').size()
eligible_test = sorted(heldout_counts[heldout_counts >= 3].index.tolist())
assert len(eligible_training) >= TRAIN_IDS + VAL_IDS
assert len(eligible_test) >= TEST_IDS
rng = np.random.default_rng(SEED)
chosen = rng.permutation(eligible_training).tolist()
test_chosen = rng.permutation(eligible_test).tolist()
partitions = {'train': chosen[:TRAIN_IDS], 'val': chosen[TRAIN_IDS:TRAIN_IDS+VAL_IDS], 'test': test_chosen[:TEST_IDS]}
DATA = PROJECT / 'runs' / RUN_ID / 'dataset_a'
DATA.mkdir(parents=True)
frames = {}
for split, identities in partitions.items():
    groups = []
    for identity in identities:
        group = pool[pool.identity == identity]
        groups.append(group.sample(n=min(MAX_IMAGES, len(group)), random_state=SEED).sort_values('image_path'))
    frame = pd.concat(groups, ignore_index=True).sort_values(['identity', 'image_path']).reset_index(drop=True)
    frame['split'] = split
    validate_images(original, frame.image_path)
    for relative in frame.image_path:
        destination = DATA / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(original / relative, destination)
    frame.to_parquet(DATA / (split + '.parquet'), index=False)
    frames[split] = frame
    print(split, len(frame), 'images;', frame.identity.nunique(), 'identities')
assert not set(frames['train'].identity) & set(frames['test'].identity)
assert not set(frames['train'].identity) & set(frames['val'].identity)
assert not set(frames['val'].identity) & set(frames['test'].identity)
MANIFEST = BACKUP / 'split_metadata'
MANIFEST.mkdir()
for path in DATA.glob('*.parquet'):
    shutil.copy2(path, MANIFEST / path.name)
CONFIG = {'seed': SEED, 'train_identities': TRAIN_IDS, 'val_identities': VAL_IDS, 'test_identities': TEST_IDS, 'max_images_per_identity': MAX_IMAGES, 'stage1_epochs':10, 'stage2_epochs':5, 'stage1_batch_size':64, 'stage1_num_workers':2, 'P':4, 'K':4, 'split_kind':'new identity-disjoint subset, not original course split', 'source_inventory':'train/val from supplied training identities; test from supplied test identities absent from all supplied training metadata'}
(BACKUP / 'experiment.json').write_text(json.dumps(CONFIG, indent=2))
CKPT = BACKUP / 'new_checkpoints'
CKPT.mkdir()
print('Split metadata saved to Drive. New checkpoints will go to:', CKPT)

## Optional: benchmark your recovered checkpoint now
This measures the saved original model on the new fixed image workload, without retraining. It makes no held-out accuracy claim. You can stop after this cell if you only need the original model's efficiency today. Keep its results separate from the newly trained model.

In [ ]:
run('scripts/measure_efficiency.py', '--model', 'clip', '--checkpoint', PROJECT / 'checkpoints/stage2_last.pt', '--dataset_root', DATA, '--dataset_name', 'dataset_a', '--device', 'cuda', '--batch_size', 16, '--num_workers', 0, '--repeats', 3, '--output_dir', BACKUP / 'original_checkpoint_efficiency')


## Stage 1: learn prompts
This downloads pretrained CLIP weights on first use. It does not use or overwrite your recovered checkpoint. The completed Stage 1 checkpoint is written directly to this run's Drive folder. If the session stops before saving, restart Stage 1; training resume is not implemented.

In [ ]:
run('train_stage1.py', '--dataset_root', DATA, '--epochs', CONFIG['stage1_epochs'], '--batch_size', CONFIG['stage1_batch_size'], '--category', 'animal', '--device', 'cuda', '--seed', SEED, '--num_workers', CONFIG['stage1_num_workers'], '--checkpoint_dir', CKPT)
assert (CKPT / 'stage1_prompts.pt').is_file()

## Stage 2: fine-tune the image encoder
Uses P=4, K=4 (batch 16) and the same training metadata. A five-epoch run is a starting experiment, not evidence of convergence. The final checkpoint is saved directly to Drive. If GPU memory runs out, stop, reduce batch size/P and restart the affected stage; record the changes in the experiment configuration before the new run. Do not change settings based on test scores.

In [ ]:
run('train_stage2.py', '--dataset_root', DATA, '--stage1_checkpoint', CKPT / 'stage1_prompts.pt', '--epochs', CONFIG['stage2_epochs'], '--warmup_epochs', 1, '--P', CONFIG['P'], '--K', CONFIG['K'], '--category', 'animal', '--device', 'cuda', '--seed', SEED, '--num_workers', 0, '--checkpoint_dir', CKPT)
assert (CKPT / 'stage2_last.pt').is_file()

## Run the fixed test comparison
Run after choosing settings. Each model uses the same held-out identities, ImageNet preprocessing, batch size 16 and full-gallery rankings. These are local project metrics, not a substitute for the official course evaluator. The recovered old checkpoint is excluded from this accuracy comparison because its training membership is unknown.

In [ ]:
REPORTS = BACKUP / 'results'
PREDICTIONS = BACKUP / 'predictions'
REPORTS.mkdir(exist_ok=True)
PREDICTIONS.mkdir(exist_ok=True)
models = [('resnet', None), ('clip_pretrained', None), ('clip_finetuned', CKPT / 'stage2_last.pt')]
for label, checkpoint in models:
    prediction = PREDICTIONS / (label + '.csv')
    args = ['models/resnet_baseline.py' if label == 'resnet' else 'predict.py', '--dataset_root', DATA, '--dataset_name', 'dataset_a', '--output', prediction, '--device', 'cuda', '--batch_size', 16, '--num_workers', 0, '--top_k', 0]
    if checkpoint:
        args += ['--checkpoint', checkpoint]
    run(*args)
    run('evaluate.py', '--student_id', label, '--prediction', prediction, '--datasets_root', DATA.parent, '--datasets', 'dataset_a', '--output_dir', REPORTS)
    efficiency = ['scripts/measure_efficiency.py', '--model', 'resnet' if label == 'resnet' else 'clip', '--dataset_root', DATA, '--dataset_name', 'dataset_a', '--device', 'cuda', '--batch_size', 16, '--num_workers', 0, '--repeats', 3, '--output_dir', REPORTS]
    if checkpoint:
        efficiency += ['--checkpoint', checkpoint]
    run(*efficiency)
print('All predictions, metrics and efficiency reports saved to', BACKUP)

## Summarize and download a second copy
Keep the Drive folder **and** a local copy. The ZIP includes source/original checkpoints, new checkpoints, exact split metadata, environment details, predictions and results; it omits image data, which remain in the separate original archive. Training logs remain in this notebook: save the notebook to Drive too.

In [ ]:
rows = []
for label, _ in models:
    reports = sorted(REPORTS.glob(label + '_*.json'))
    quality = json.loads(reports[-1].read_text())['datasets']['dataset_a']['performance']
    name = {'resnet':'resnet50_zero_shot', 'clip_pretrained':'clip_reid_zeroshot', 'clip_finetuned':'clip_reid_finetuned'}[label]
    efficiency = json.loads((REPORTS / ('efficiency_' + name + '_cuda.json')).read_text())
    rows.append({'model':label, 'Rank-1':quality['Rank-1'], 'mAP':quality['mAP'], 'mINP':quality['mINP'], 'images/sec':efficiency['throughput_images_per_sec'], 'peak allocated MiB':max(efficiency['peak_memory_mb_after_query'], efficiency['peak_memory_mb_after_gallery']), 'embedding_dim':efficiency['embedding_dim'], 'total_eval_sec':efficiency['total_eval_time_sec']})
summary = pd.DataFrame(rows)
summary.to_csv(BACKUP / 'comparison.csv', index=False)
display(summary)
archive_path = shutil.make_archive('/content/obj_reid_results_' + RUN_ID, 'zip', BACKUP)
from google.colab import files
files.download(archive_path)
print('Also keep the saved notebook and Drive run folder:', BACKUP)